In [20]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

In [6]:
def smiles_to_morgan_fp(smiles: str, n_bits: int = 1024, radius: int = 2) -> np.ndarray:
    """
    Converts a SMILES string to a Morgan fingerprint.
    """
    mol = Chem.MolFromSmiles(smiles)    
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

def show_random_molecules(smiles_list, N=5):
    """
    Given a list of smiles, returns N visualizations of molecules
    """
    sampled_smiles = random.sample(smiles_list, min(N, len(smiles_list)))
    mols = [Chem.MolFromSmiles(smi) for smi in sampled_smiles]
    img = Draw.MolsToGridImage(mols, molsPerRow=10, subImgSize=(200, 200), legends=sampled_smiles)
    display(img)

**REAL NEGATIVES**

In [7]:
df = pd.read_csv("/home/acomajuncosa/Downloads/activities_CHEMBL1794345.tsv", sep='\t', low_memory=False)
df = df[~df['Smiles'].isna()]
print("No pvalues: " + str([i for i in df['pChEMBL Value'].tolist() if not np.isnan(i)]))
print("All are nM: " + str(set([i for i in df['Standard Units']])))

# Generate pChEMBLs
df['pChEMBL_calculated'] = [-np.log10(i * 1e-09) for i in df['Standard Value']]

# Get actives and inactives
actives = df[df['pChEMBL_calculated'] >= 7]['Smiles'].tolist()
inactives = df[df['pChEMBL_calculated'] < 7]['Smiles'].tolist()
print("Actives: " + str(len(actives)))
print("Inactives: " + str(len(inactives)))

# Fix random seed
np.random.seed(42)

# Choose N actives and 1 * N inactives
# N = 7000
# selected_actives = np.random.choice(actives, N, replace=False).tolist()
# selected_inactives = np.random.choice(inactives, 1 * N, replace=False).tolist()
selected_actives = actives
selected_inactives = inactives
print("Actives: " + str(len(selected_actives)))
print("Inactives: " + str(len(selected_inactives)))

# Get ECFPs
print("Calculating ECFPs...")
actives_smiles = list(selected_actives)
inactives_smiles = list(selected_inactives)
selected_actives = [smiles_to_morgan_fp(i) for i in selected_actives]
selected_inactives = [smiles_to_morgan_fp(i) for i in selected_inactives]

# Create matrices
X = np.array(selected_actives + selected_inactives)
Y = np.array([1]*len(selected_actives) + [0]*len(selected_inactives))
print("Matrix shapes:")
print(X.shape, Y.shape)

No pvalues: []
All are nM: {'nM'}
Actives: 7364
Inactives: 162948
Actives: 7364
Inactives: 162948
Calculating ECFPs...


[17:09:17] WARNING: not removing hydrogen atom without neighbors


Matrix shapes:
(170312, 1024) (170312,)


In [55]:
def segment_and_evaluate(
    X, 
    y, 
    base_model=None, 
    max_depth=5, 
    min_samples_leaf=10000, 
    cv=5
):
    """
    1) Trains a shallow DecisionTreeClassifier on (X, y).
    2) Uses the tree to segment the dataset into leaves.
    3) Within each leaf, trains/evaluates a base model via cross-validation (AUROC).
    4) Returns a DataFrame of leaves that meet the specified AUROC threshold.

    Parameters
    ----------
    X : pd.DataFrame or np.ndarray
        Feature matrix.
    y : pd.Series or np.ndarray
        Target array.
    base_model : sklearn estimator, optional
        The base model to evaluate on each leaf. Defaults to LogisticRegression.
    max_depth : int
        Maximum depth of the decision tree used for segmentation.
    min_samples_leaf : int
        Minimum number of samples required in each leaf of the tree.
    cv : int
        Number of cross-validation folds to use for evaluating the base model.
    auc_threshold : float
        Only return leaves whose AUROC >= this threshold.

    Returns
    -------
    pd.DataFrame
        A table where each row corresponds to a leaf that meets the threshold.
        Columns include:
          - leaf_index: The integer ID of the leaf.
          - n_samples: Number of samples in that leaf.
          - auc: The cross-validated AUROC of the base model on that leaf.
          - rule: The text representation of the entire tree (optional, for interpretability).
    """

    if base_model is None:
        base_model = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=8)

    # 1) Fit a shallow decision tree
    tree = DecisionTreeClassifier(
        max_depth=max_depth, 
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    tree.fit(X, y)

    print("Decision tree fitted!")

    # 2) Assign each sample to a leaf
    leaf_indices = tree.apply(X)
    unique_leaves = np.unique(leaf_indices)

    print(f"{len(unique_leaves)} unique leaves")

    # Prepare a list to hold performance results for each leaf
    leaf_results = []

    # 3) For each leaf, train/evaluate the base model using cross-validation
    for leaf_id in unique_leaves:

        print(f"Leaf ID: {leaf_id}")

        mask = (leaf_indices == leaf_id)
        X_subset = X[mask]
        y_subset = y[mask]

        # Cross-validation on this subset
        model = clone(base_model)
        auc_scores = cross_val_score(
            model, X_subset, y_subset, 
            cv=cv, 
            scoring='roc_auc'
        )
        auc_mean = np.mean(auc_scores)
        auc_std = np.std(auc_scores)

        # Record the results
        leaf_results.append({
            'leaf_index': leaf_id,
            'n_samples': len(X_subset),
            'auc mean': auc_mean,
            'auc std': auc_std})
            # This is optional, but can help interpret how the tree split the data
            # 'rule': export_text(tree, feature_names=[i for i in range(X.shape[1])])  

    # 4) Create a DataFrame, then filter leaves by AUROC threshold
    results_df = pd.DataFrame(leaf_results)

    return results_df

In [ ]:
results = segment_and_evaluate(
    X=X[:50000],
    y=Y[:50000],
    base_model=RandomForestClassifier(n_estimators=100),
    max_depth=7,
    min_samples_leaf=80,
    cv=4
)

print("Leaves with AUROC >= 0.75:")
print(results)

Decision tree fitted!
53 unique leaves
Leaf ID: 7
Leaf ID: 8


In [53]:
results

,leaf_index,n_samples,auc mean,auc std
0,3,31090,0.520501,0.007875
1,4,5779,0.524279,0.025127
2,5,3019,0.561743,0.010425
3,8,2829,0.522098,0.032458
4,9,3173,0.512531,0.036360
5,11,2107,0.507081,0.030816
6,12,2003,0.550370,0.028905
